# Final Report Figure Reproduction

This notebook reproduces the seven final report figures. When the original local project artifacts are present next to the repository, it uses the same generated outputs and QA artifacts used by `project_showcase.ipynb`. When those private or large artifacts are absent, it falls back to the reduced aggregate fixtures in `data/fixtures/`.

The fixture fallback contains only aggregate metrics and scalar logs: no source document text, reference summaries, full predictions, API keys, or model weights.

## Setup

In [ ]:
from __future__ import annotations

import json
import math
import statistics
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = Path.cwd().parent
ARCHIVE_ROOT = ROOT.parent
FIG_DIR = ROOT / "figures"
FIXTURE_DIR = ROOT / "data" / "fixtures"
FIG_DIR.mkdir(exist_ok=True)

METRICS = json.loads((FIXTURE_DIR / "final_report_metrics.json").read_text())
TRAINER_STATE = json.loads((FIXTURE_DIR / "trainer_state.json").read_text())

SOURCE_PATHS = {
    "test_split": ARCHIVE_ROOT / "First_Train" / "test.jsonl",
    "trainer_state": ARCHIVE_ROOT / "Polish" / "checkpoint-3690" / "trainer_state.json",
    "ckpt3000_scored": ARCHIVE_ROOT / "eval_ckpt3000_scored.csv",
    "ckpt3690_scored": ARCHIVE_ROOT / "Polish" / "Eval2" / "eval_ckpt3690_scored.csv",
    "claude_scored": ARCHIVE_ROOT / "eval_claude_code_scored.csv",
    "metadata_eval": ARCHIVE_ROOT / "final-additions" / "metadata-baseline" / "outputs" / "eval_meta_baseline_qwen7b.jsonl",
    "qa_ckpt3000": ARCHIVE_ROOT / "Polish" / "Eval2" / "qa_report_ckpt3000" / "qa_report.jsonl",
    "qa_ckpt3690": ARCHIVE_ROOT / "Polish" / "Eval2" / "qa_report_ckpt3690" / "qa_report.jsonl",
    "qa_metadata": ARCHIVE_ROOT / "final-additions" / "metadata-baseline" / "outputs" / "qa_qwen7b_meta" / "qa_report.jsonl",
    "qa_claude": ARCHIVE_ROOT / "qa_report_claude_code" / "qa_report.jsonl",
    "attr_ckpt3000": ARCHIVE_ROOT / "final-additions" / "attribution" / "outputs" / "attribution_ckpt3000.jsonl",
    "attr_ckpt3690": ARCHIVE_ROOT / "final-additions" / "attribution" / "outputs" / "attribution_ckpt3690.jsonl",
    "attr_metadata": ARCHIVE_ROOT / "final-additions" / "attribution" / "outputs" / "attribution_meta_qwen7b.jsonl",
}
USE_ORIGINAL_ARTIFACTS = all(path.exists() for path in SOURCE_PATHS.values())
print("Using original project_showcase artifacts:" if USE_ORIGINAL_ARTIFACTS else "Using reduced public fixtures:", USE_ORIGINAL_ARTIFACTS)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})

BLUE = "#4C78A8"
ORANGE = "#F58518"
GREEN = "#54A24B"
RED = "#E45756"
PURPLE = "#B279A2"
GRAY = "#6E6E6E"
AMBER = "#ECA400"

def read_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

def save_figure(fig, filename: str):
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, bbox_inches="tight")
    print("wrote", FIG_DIR / filename)


## Figure 1: Training Dynamics

In [ ]:
# Figure 1: training dynamics from the final checkpoint trainer log.
if USE_ORIGINAL_ARTIFACTS:
    raw_state = json.loads(SOURCE_PATHS["trainer_state"].read_text())
    log_history = [row for row in raw_state["log_history"] if "loss" in row]
else:
    log_history = TRAINER_STATE["log_history"]

train_df = pd.DataFrame(log_history).sort_values("step")
ema = train_df["loss"].ewm(alpha=0.1, adjust=False).mean()
ema_min_idx = int(ema.idxmin())
ema_min_step = int(train_df.loc[ema_min_idx, "step"])
final_step = int(train_df["step"].max())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(train_df["step"], train_df["loss"], color=BLUE, alpha=0.35, linewidth=1, label="raw loss")
axes[0].plot(train_df["step"], ema, color=BLUE, linewidth=2.2, label="EMA loss")
axes[0].axvline(ema_min_step, color=GREEN, linestyle="--", linewidth=1.5, label=f"EMA min: {ema_min_step}")
axes[0].axvline(final_step, color=RED, linestyle="--", linewidth=1.5, label=f"final: {final_step}")
axes[0].set_title("Training loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].legend(loc="best")
axes[0].grid(alpha=0.25)

axes[1].plot(train_df["step"], train_df["mean_token_accuracy"], color=GREEN, linewidth=2)
axes[1].axvline(ema_min_step, color=GREEN, linestyle="--", linewidth=1.2)
axes[1].axvline(final_step, color=RED, linestyle="--", linewidth=1.2)
axes[1].set_title("Token accuracy")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Mean token accuracy")
axes[1].grid(alpha=0.25)

axes[2].plot(train_df["step"], train_df["learning_rate"], color=PURPLE, linewidth=2)
axes[2].axvline(ema_min_step, color=GREEN, linestyle="--", linewidth=1.2)
axes[2].axvline(final_step, color=RED, linestyle="--", linewidth=1.2)
axes[2].set_title("Learning-rate schedule")
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Learning rate")
axes[2].grid(alpha=0.25)

save_figure(fig, "figure1_training_dynamics.png")


## Figure 2: Prompt Length Distribution

In [ ]:
# Figure 2: prompt-size distribution from First_Train/test.jsonl when available.
if USE_ORIGINAL_ARTIFACTS:
    prompt_tokens = sorted(len(row.get("prompt", "")) / 4 for row in read_jsonl(SOURCE_PATHS["test_split"]))
    fit_32k = sum(v <= 32_000 for v in prompt_tokens) / len(prompt_tokens) * 100
    fit_200k = sum(v <= 200_000 for v in prompt_tokens) / len(prompt_tokens) * 100
else:
    bins = METRICS["prompt_length_distribution"]["bins"]
    prompt_tokens = []
    for bucket in bins:
        low = bucket["min_tokens"]
        high = bucket["max_tokens"] or 256_000
        prompt_tokens.extend(np.linspace(max(low, 500), high, bucket["count"]).tolist())
    prompt_tokens = sorted(prompt_tokens)
    fit_32k = METRICS["prompt_length_distribution"]["fit_32k_pct"]
    fit_200k = METRICS["prompt_length_distribution"]["fit_200k_pct"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
axes[0].hist(prompt_tokens, bins=np.logspace(np.log10(500), np.log10(max(prompt_tokens)), 45), color=BLUE, alpha=0.82)
axes[0].axvline(32_000, color=GREEN, linestyle="--", label="32K context")
axes[0].axvline(200_000, color=RED, linestyle="--", label="200K context")
axes[0].set_xscale("log")
axes[0].set_title("Estimated prompt size distribution")
axes[0].set_xlabel("Estimated tokens (characters / 4)")
axes[0].set_ylabel("Cases")
axes[0].legend()
axes[0].grid(alpha=0.2)

y = np.arange(1, len(prompt_tokens) + 1) / len(prompt_tokens) * 100
axes[1].plot(prompt_tokens, y, color=BLUE, linewidth=2)
axes[1].axvline(32_000, color=GREEN, linestyle="--", label=f"32K: {fit_32k:.1f}% fit")
axes[1].axvline(200_000, color=RED, linestyle="--", label=f"200K: {fit_200k:.1f}% fit")
axes[1].set_xscale("log")
axes[1].set_ylim(0, 100)
axes[1].set_title("Cumulative coverage by context window")
axes[1].set_xlabel("Estimated tokens")
axes[1].set_ylabel("Percent of test cases")
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.2)

save_figure(fig, "figure2_prompt_length_distribution.png")


## Figure 3: Checkpoint Comparison

In [ ]:
# Figure 3: checkpoint metric comparison from the scored CSVs / reduced fixture.
if USE_ORIGINAL_ARTIFACTS:
    ckpt3000 = pd.read_csv(SOURCE_PATHS["ckpt3000_scored"]).mean(numeric_only=True).to_dict()
    ckpt3690 = pd.read_csv(SOURCE_PATHS["ckpt3690_scored"]).mean(numeric_only=True).to_dict()
else:
    ckpt3000 = METRICS["systems"]["ckpt-3000"]
    ckpt3690 = METRICS["systems"]["ckpt-3690"]

metrics_to_plot = [("rouge1", "ROUGE-1"), ("rouge2", "ROUGE-2"), ("rougeL", "ROUGE-L"), ("rougeLsum", "ROUGE-Lsum"), ("bertscore_f1", "BERTScore F1")]
x = np.arange(len(metrics_to_plot))
width = 0.36
fig, ax = plt.subplots(figsize=(10, 5))
y3000 = [ckpt3000[k] for k, _ in metrics_to_plot]
y3690 = [ckpt3690[k] for k, _ in metrics_to_plot]
bars1 = ax.bar(x - width / 2, y3000, width, label="ckpt-3000", color=BLUE)
bars2 = ax.bar(x + width / 2, y3690, width, label="ckpt-3690", color=ORANGE)
ax.set_title("Checkpoint comparison: ROUGE and BERTScore")
ax.set_ylabel("Mean score on same 50 cases")
ax.set_xticks(x)
ax.set_xticklabels([label for _, label in metrics_to_plot], rotation=20, ha="right")
ax.legend()
ax.grid(axis="y", alpha=0.25)
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.006, f"{h:.3f}", ha="center", va="bottom", fontsize=8)

save_figure(fig, "figure3_checkpoint_comparison.png")


## Figure 4: QA Triage

In [ ]:
# Figure 4: QA triage status from the QA reports / reduced fixture.
def qa_counts(path):
    counts = Counter(row.get("status", "UNKNOWN") for row in read_jsonl(path))
    total = sum(counts.values())
    return {status: counts.get(status, 0) / total * 100 for status in ["PASS", "REVIEW", "REJECT"]}

if USE_ORIGINAL_ARTIFACTS:
    qa = {
        "ckpt-3000": qa_counts(SOURCE_PATHS["qa_ckpt3000"]),
        "ckpt-3690": qa_counts(SOURCE_PATHS["qa_ckpt3690"]),
        "Qwen7B metadata-only": qa_counts(SOURCE_PATHS["qa_metadata"]),
        "Claude Code (Sonnet 4.6)": qa_counts(SOURCE_PATHS["qa_claude"]),
    }
else:
    qa = {system: data["percent"] for system, data in METRICS["qa_triage"].items()}

status_order = ["PASS", "REVIEW", "REJECT"]
system_order = ["ckpt-3000", "ckpt-3690", "Qwen7B metadata-only", "Claude Code (Sonnet 4.6)"]
x = np.arange(len(system_order))
width = 0.24
fig, ax = plt.subplots(figsize=(11, 5))
colors = {"PASS": GREEN, "REVIEW": AMBER, "REJECT": RED}
for i, status in enumerate(status_order):
    vals = [qa[system][status] for system in system_order]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=status, color=colors[status])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 1.2, f"{val:.0f}%", ha="center", va="bottom", fontsize=8)
ax.set_title("QA triage status by system")
ax.set_ylabel("Percent of 50-case sample")
ax.set_ylim(0, 82)
ax.set_xticks(x)
ax.set_xticklabels(system_order, rotation=18, ha="right")
ax.legend(ncol=3, loc="upper left")
ax.grid(axis="y", alpha=0.25)

save_figure(fig, "figure4_qa_triage_3systems.png")


## Figure 5: Source Attribution

In [ ]:
# Figure 5: source attribution coverage from attribution JSONL / reduced fixture.
def attr_stats(path):
    rows = read_jsonl(path)
    return {
        "primary_pct": statistics.mean(row.get("coverage_primary", 0) for row in rows) * 100,
        "supported_pct": statistics.mean(row.get("coverage_supported", 0) for row in rows) * 100,
    }

if USE_ORIGINAL_ARTIFACTS:
    attribution = {
        "ckpt-3000": attr_stats(SOURCE_PATHS["attr_ckpt3000"]),
        "ckpt-3690": attr_stats(SOURCE_PATHS["attr_ckpt3690"]),
        "Qwen7B metadata-only": attr_stats(SOURCE_PATHS["attr_metadata"]),
    }
else:
    attribution = METRICS["source_attribution"]

attr_systems = ["ckpt-3000", "ckpt-3690", "Qwen7B metadata-only"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
primary = [attribution[s]["primary_pct"] for s in attr_systems]
supported = [attribution[s]["supported_pct"] for s in attr_systems]
axes[0].bar(attr_systems, primary, color=[BLUE, ORANGE, GREEN])
axes[0].set_title("Primary source coverage")
axes[0].set_ylabel("Mean sentence coverage per case (%)")
axes[0].set_ylim(0, 105)
axes[0].tick_params(axis="x", rotation=18)
axes[0].grid(axis="y", alpha=0.25)
axes[1].bar(attr_systems, supported, color=[BLUE, ORANGE, GREEN])
axes[1].set_title("Primary + weak support coverage")
axes[1].set_ylim(0, 105)
axes[1].tick_params(axis="x", rotation=18)
axes[1].grid(axis="y", alpha=0.25)
for ax, vals in zip(axes, [primary, supported]):
    for idx, val in enumerate(vals):
        ax.text(idx, val + 1.5, f"{val:.1f}%", ha="center", va="bottom", fontsize=9)

save_figure(fig, "figure5_source_attribution.png")


## Figure 6: Cost vs. Quality

In [ ]:
# Figure 6: cost-quality tradeoff from project_showcase framing.
rows = METRICS["cost_quality"]
fig, ax = plt.subplots(figsize=(9, 5.2))
colors = [BLUE, GREEN, PURPLE, RED]
label_offsets = {
    "ckpt-3000 (self-hosted)": (8, 10, "left"),
    "Qwen7B metadata-only": (8, -13, "left"),
    "Claude Code Sonnet 4.6 (subscription)": (8, 0, "left"),
    "Claude Sonnet API estimate": (-8, 0, "right"),
}
for row, color in zip(rows, colors):
    label = row["system"].replace(" (self-hosted)", "").replace("Claude Code Sonnet 4.6 (subscription)", "Claude Code")
    ax.scatter(row["cost_per_case_usd"], row["rouge1"], s=95, color=color, edgecolor="black", linewidth=0.6, zorder=3)
    dx, dy, ha = label_offsets[row["system"]]
    ax.annotate(label, (row["cost_per_case_usd"], row["rouge1"]), xytext=(dx, dy), textcoords="offset points", ha=ha, va="center", fontsize=9)
ax.set_title("Cost-quality tradeoff on the 50-case sample")
ax.set_xlabel("Estimated marginal cost per case (USD)")
ax.set_ylabel("Mean ROUGE-1")
ax.set_xlim(-0.006, 0.092)
ax.set_ylim(0.34, 0.485)
ax.set_xticks([0, 0.02, 0.04, 0.06, 0.08])
ax.set_xticklabels(["$0", "$0.02", "$0.04", "$0.06", "$0.08"])
ax.grid(alpha=0.25)
ax.text(0.001, 0.345, "Claude Code was measured through subscription; API point is an estimate from the same outputs.", fontsize=8.5, color=GRAY)

save_figure(fig, "figure6_cost_quality.png")


## Figure 7: QA Flag Frequency

In [ ]:
# Figure 7: ckpt-3690 QA flag frequency.
def top_flags(path, top_n=12):
    counts = Counter()
    severities = {}
    for row in read_jsonl(path):
        for flag in row.get("flags", []):
            code = flag.get("code", "UNKNOWN")
            counts[code] += 1
            severities.setdefault(code, flag.get("severity", "info"))
    return [{"code": code, "count": count, "severity": severities.get(code, "info")} for code, count in counts.most_common(top_n)]

flags = top_flags(SOURCE_PATHS["qa_ckpt3690"]) if USE_ORIGINAL_ARTIFACTS else METRICS["ckpt3690_flag_frequency"]
fig, ax = plt.subplots(figsize=(10, 5.6))
flags = list(reversed(flags))
severity_colors = {"critical": RED, "warning": AMBER, "info": BLUE}
bar_colors = [severity_colors.get(row["severity"], GRAY) for row in flags]
ypos = np.arange(len(flags))
ax.barh(ypos, [row["count"] for row in flags], color=bar_colors)
ax.set_yticks(ypos)
ax.set_yticklabels([row["code"] for row in flags])
ax.set_xlabel("Flag count across 50 ckpt-3690 summaries")
ax.set_title("Most common QA flags in the overfit checkpoint")
ax.grid(axis="x", alpha=0.25)
for yv, row in zip(ypos, flags):
    ax.text(row["count"] + 0.8, yv, str(row["count"]), va="center", fontsize=9)
handles = [plt.Line2D([0], [0], color=severity_colors[s], lw=6) for s in ["critical", "warning", "info"]]
ax.legend(handles, ["critical", "warning", "info"], loc="lower right")

save_figure(fig, "figure7_flag_frequency.png")


## Verify Outputs

In [ ]:
expected = [
    "figure1_training_dynamics.png",
    "figure2_prompt_length_distribution.png",
    "figure3_checkpoint_comparison.png",
    "figure4_qa_triage_3systems.png",
    "figure5_source_attribution.png",
    "figure6_cost_quality.png",
    "figure7_flag_frequency.png",
]
for name in expected:
    path = FIG_DIR / name
    print(name, path.exists(), f"{path.stat().st_size:,} bytes" if path.exists() else "missing")
